In [1]:
import networkx as nx;
import pandas as pd;
import gurobipy as gp;
from gurobipy import GRB;
import csv;
import sys;
import numpy

In [2]:
networkCSV = 'TestInstances/CSV_TestInstances/N100/' + 'N100_22.csv';
N = 1000; #sample size
budget = 5;
option = 2; #1: conservative refinement; 2: aggressive refinement
numpy.random.seed(2024);

In [3]:
# Reading network file
with open(networkCSV, newline='') as f:
    reader = csv.reader(f);
    row1 = next(reader);
    nbArcs = int(row1[0]);
    row2 = next(reader);
    s = int(row2[0]);
    row3 = next(reader);
    t = int(row3[0]);
    
    G = nx.DiGraph();
    data = pd.read_csv(networkCSV, skiprows=4, header=None, delim_whitespace=True);
    n_edge = len(data.index);

    for i in range(n_edge): 
        G.add_edge(data.iat[i,0], data.iat[i,1], costLB = data.iat[i,2], 
                costUB = data.iat[i,3], interEffect = data.iat[i,4], tempCost = 0);

In [4]:
'''
print("nbArcs = ", nbArcs);
print("s = ", s);
print("t = ", t);
print("G.nodes = ", G.nodes)
print("G.edges = ", G.edges)
for e in G.edges:
    print(e)
    print(G.edges[e])
'''

'\nprint("nbArcs = ", nbArcs);\nprint("s = ", s);\nprint("t = ", t);\nprint("G.nodes = ", G.nodes)\nprint("G.edges = ", G.edges)\nfor e in G.edges:\n    print(e)\n    print(G.edges[e])\n'

In [5]:
# Creating samples
scens = [];
for k in range(N):
    scen = {};
    for e in G.edges:
        scen[e] = numpy.random.uniform(G.edges[e]['costLB'],G.edges[e]['costUB']);
    scens.append(scen);

In [6]:
def findCluster(spPath, k, clusters, clusterPaths):
    if len(clusters) == 0:
        clusters.append([k]);
        clusterPaths.append(spPath);
    else:
        flag = False;
        for l in range(len(clusters)):
            if len(spPath) == len(clusterPaths[l]):
                flag = True;
                for m in range(len(spPath)):
                    if spPath[m] != clusterPaths[l][m]:
                        flag = False;
                        break;
                if flag:
                    clusters[l].append(k);
                    break;
        if not flag:
            clusters.append([k]);
            clusterPaths.append(spPath);
    return clusters, clusterPaths

In [7]:
# Callback - use lazy constraints
def lazy(model, where):
    if where == GRB.Callback.MIPSOL:
        xvals = model.cbGetSolution(model._x)
        thetaval = model.cbGetSolution(model._theta);
        # Phase-1: Just using the existing partition -- coarse cuts
        partitionCosts = [];
        partitionPaths = [];
        #partitionEdgeCosts = [];
        totalCost = 0;
        for p in range(len(model._partition)):
            # update edge cost per partition
            #partitionEdgeCost = {};
            for e in model._G.edges:
                #edgeCost = 0;
                #for k in model._partition[p]:
                #    edgeCost += model._scens[k][e];
                #edgeCost = edgeCost*1.0/len(model._partition[p]);
                #partitionEdgeCost[e] = edgeCost;
                if xvals[e] > 1e-5:
                    model._G.edges[e]['tempCost'] = model._partition[p]["edgeCosts"][e] + model._G.edges[e]['interEffect'];
                else:
                    model._G.edges[e]['tempCost'] = model._partition[p]["edgeCosts"][e];
            # obtain the shortest path and its length
            spValue = nx.shortest_path_length(model._G, source=model._s, target=model._t, weight='tempCost', method='dijkstra')
            spPath = nx.shortest_path(model._G, source=model._s, target=model._t, weight='tempCost', method='dijkstra')
            partitionCosts.append(spValue); 
            partitionPaths.append(spPath);
            #partitionEdgeCosts.append(partitionEdgeCost);
            totalCost += spValue*len(model._partition[p]["scenList"])/N;
        if totalCost < thetaval-(1e-5):
            # add lazy constraints
            constrCoefList = [1];
            constrVarList = [model._theta];
            rhs = 0;
            for p in range(len(model._partition)):
                for i in range(len(partitionPaths[p])-1):
                    rhs += model._partition[p]["edgeCosts"][(partitionPaths[p][i],partitionPaths[p][i+1])]*len(model._partition[p]["scenList"])/N;
                    constrCoefList.append(-model._G.edges[(partitionPaths[p][i],partitionPaths[p][i+1])]['interEffect']*len(model._partition[p]["scenList"])/N);
                    constrVarList.append(model._x[(partitionPaths[p][i],partitionPaths[p][i+1])]);
            expr = gp.LinExpr();
            expr.addTerms(constrCoefList, constrVarList);
            model.cbLazy(expr <= rhs);
        else:
            # Phase-2: Refine into semi-coarse cuts or fine cuts
            # First sort the partition list from the largest to smallest
            sortedList = sorted(range(len(model._partition)), key=lambda k: len(model._partition[k]["scenList"]), reverse = True);
            newPartition = [];
            newPartitionPaths = [];
            newPartitionEdgeCosts = [];
            # Now start refining
            flag = True;
            for p in sortedList:
                fflag = False;
                if model._option == 1:
                    # conservative refinement strategy
                    if flag and len(model._partition[p]["scenList"]) > 1:
                        fflag = True;
                if model._option == 2:
                    # aggressive refinement strategy
                    if len(model._partition[p]["scenList"]) > 1:
                        fflag = True;
                if fflag: 
                    totalCost -= partitionCosts[p]*len(model._partition[p]["scenList"])/N;
                    clusters = [];
                    clusterPaths = [];
                    clusterCost = 0;
                    for k in model._partition[p]["scenList"]:
                        # update edge cost per scenario
                        for e in model._G.edges:
                            if xvals[e] > 1e-5:
                                model._G.edges[e]['tempCost'] = model._scens[k][e] + model._G.edges[e]['interEffect'];
                            else:
                                model._G.edges[e]['tempCost'] = model._scens[k][e];
                        # obtain the shortest path and its length
                        spValue = nx.shortest_path_length(model._G, source=model._s, target=model._t, weight='tempCost', method='dijkstra')
                        clusterCost += spValue;
                        spPath = nx.shortest_path(model._G, source=model._s, target=model._t, weight='tempCost', method='dijkstra');
                        clusters, clusterPaths = findCluster(spPath, k, clusters, clusterPaths);
                    totalCost += clusterCost*1.0/N;
                    for k in range(len(clusters)):
                        newPartitionPaths.append(clusterPaths[k]);
                        partitionEdgeCost = {};
                        for e in model._G.edges:
                            edgeCost = 0;
                            for kk in clusters[k]:
                                edgeCost += model._scens[kk][e];
                            edgeCost = edgeCost*1.0/len(clusters[k]);
                            partitionEdgeCost[e] = edgeCost;
                        newPartition.append({"scenList": clusters[k], "edgeCosts": partitionEdgeCost});
                    if totalCost < thetaval-(1e-5):
                        flag = False;
                else:
                    newPartition.append(model._partition[p]);
                    newPartitionPaths.append(partitionPaths[p]);
            if not flag: 
                # add lazy constraints
                constrCoefList = [1];
                constrVarList = [model._theta];
                rhs = 0;
                for p in range(len(newPartition)):
                    for i in range(len(newPartitionPaths[p])-1):
                        rhs += newPartition[p]["edgeCosts"][(newPartitionPaths[p][i],newPartitionPaths[p][i+1])]*len(newPartition[p]["scenList"])/N;
                        constrCoefList.append(-model._G.edges[(newPartitionPaths[p][i],newPartitionPaths[p][i+1])]['interEffect']*len(newPartition[p]["scenList"])/N);
                        constrVarList.append(model._x[(newPartitionPaths[p][i],newPartitionPaths[p][i+1])]);
                expr = gp.LinExpr();
                expr.addTerms(constrCoefList, constrVarList);
                model.cbLazy(expr <= rhs);
            model._partition = newPartition;

In [8]:
master = gp.Model()

# Create variables
x = {};
for e in G.edges:
    x[e] = master.addVar(obj=0, vtype=GRB.BINARY);

theta = master.addVar(obj=1.0, vtype=GRB.CONTINUOUS, lb = 0, ub = 1e7);

# Add interdiction budget constraint
master.addConstr(gp.quicksum(x[e] for e in G.edges) <= budget);

master._x = x
master._theta = theta
master._G = G
master._s = s
master._t = t
master._option = option
partitionEdgeCost = {};
for e in G.edges:
    edgeCost = 0;
    for k in range(N):
        edgeCost += scens[k][e];
    edgeCost = edgeCost*1.0/N;
    partitionEdgeCost[e] = edgeCost;
initial_partition = {"scenList": list(range(N)), "edgeCosts": partitionEdgeCost};
master._partition = [initial_partition]; # initial partition: putting everything together
master._scens = scens

Set parameter Username
Academic license - for non-commercial use only - expires 2024-07-30


In [9]:
master.modelSense = GRB.MAXIMIZE
master.Params.LazyConstraints = 1
master.optimize(lazy)

xvals = master.getAttr('X', x)

print('')
print('Optimal objval: %g' % master.ObjVal)
print('')
print('Optimal xval = ')
for e in G.edges:
    if xvals[e] > 1e-5:
        print(e);
        print(" ")

Set parameter LazyConstraints to value 1
Gurobi Optimizer version 9.5.0 build v9.5.0rc5 (mac64[x86])
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 1 rows, 1000 columns and 999 nonzeros
Model fingerprint: 0xf0823b2b
Variable types: 1 continuous, 999 integer (999 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+07]
  RHS range        [5e+00, 5e+00]
Presolve time: 0.00s
Presolved: 1 rows, 1000 columns, 999 nonzeros
Variable types: 1 continuous, 999 integer (999 binary)

Root relaxation: objective 1.025190e+03, 5 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 1011.28554    0    2          - 1011.28554      -     -    0s
H    0     0                     972.2216639 1011.28554  4.02%     -    3s
H    0 

In [10]:
print("# of final partitions = %d, which is %.1f %% of the total scenarios" % (len(master._partition), len(master._partition)*100.0/N) );

# of final partitions = 267, which is 26.7 % of the total scenarios
